# Titanic Survival Prediction: Classical ML Pipeline
## Week 1 Deliverable — Clean Code & SOLID Architecture

This notebook is structured to mirror the production modular package in `src/`:
- **Single Responsibility Principle (SRP)**: Each section handles one dedicated stage.
- **Open/Closed Principle (OCP)**: Candidate models and hyperparameter grids are registered without changing downstream evaluation loops.
- **Liskov Substitution Principle (LSP)**: All models conform to scikit-learn's `BaseEstimator` interface.
- **Dependency Inversion Principle (DIP)**: Pipeline orchestration depends on abstract interfaces and injected parameters.

### Section 0: Environment Setup & Global Imports
> **SRP**: Dedicated to importing dependencies and configuring global seeds.

In [ ]:
# TODO: Import standard libraries (os, sys, warnings)
# TODO: Import data manipulation libraries (pandas, numpy)
# TODO: Import visualization libraries (matplotlib.pyplot, seaborn)
# TODO: Import scikit-learn modules (model_selection, metrics, linear_model, ensemble, tree, pipeline, preprocessing)
# TODO: Set random seed = 42 for reproducibility

### Section 1: Data Ingestion & Hygiene Audit (`src/preprocessing.py`)
> **SRP**: Ingest raw data, inspect schema, and drop uninformative identifiers (`PassengerId`, `Ticket`, `Cabin`).

In [ ]:
# TODO 1.1: Load train.csv and test.csv from data directory
# TODO 1.2: Cache test 'PassengerId' for future submissions
# TODO 1.3: Inspect shapes, info, and missing values
# TODO 1.4: Drop 'PassengerId', 'Ticket', and 'Cabin' (>77% missing)

### Section 2: Statistical Imputation (`src/preprocessing.py`)
> **SRP & Data Leakage Prevention**: Fit medians/modes on the training distribution ONLY, then transform both train and test.

In [ ]:
# TODO 2.1: Impute numerical continuous 'Age' with training median (robust against skew/outliers)
# TODO 2.2: Impute numerical continuous 'Fare' with training median
# TODO 2.3: Impute categorical nominal 'Embarked' with training mode ('S')
# TODO 2.4: Verify zero null values remain

### Section 3: Domain Feature Engineering & Categorical Encoding (`src/preprocessing.py`)
> **SRP**: Extract predictive signals from raw features without introducing target leakage.

In [ ]:
# TODO 3.1: Extract 'Title' from passenger 'Name' using regex r' ([A-Za-z]+)\.'
# TODO 3.2: Group rare titles ('Dr', 'Rev', 'Col', etc.) into 'Rare'
# TODO 3.3: Engineer 'FamilySize' = SibSp + Parch + 1
# TODO 3.4: Engineer binary 'IsAlone' = (FamilySize == 1).astype(int)
# TODO 3.5: Drop raw 'Name' column
# TODO 3.6: Encode binary 'Sex' (female -> 1, male -> 0)
# TODO 3.7: One-hot encode nominal categories ('Embarked', 'Title') with drop_first=True

### Section 4: Stratified Splitting (`src/preprocessing.py`)
> **Stratification Rule**: Preserve the ~61.6% (Perished) : 38.4% (Survived) class ratio across train (80%) and held-out validation (20%).

In [ ]:
# TODO 4.1: Separate feature matrix X and target vector y ('Survived')
# TODO 4.2: Perform train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# TODO 4.3: Verify train shape (712 rows) and validation shape (179 rows)
# TODO 4.4: Check class balance percentages in both splits to confirm stratification

### Section 5: Model Registry & Cross-Validation Arena (`src/train.py` & `src/evaluate.py`)
> **OCP & LSP**: Instantiate candidate models conforming to `BaseEstimator`. Benchmark using 5-Fold `StratifiedKFold` and compute the Overfit Gap $\Delta = \text{Train} - \text{Val}$.

In [ ]:
# TODO 5.1: Define model registry dictionary (Dummy, LogisticRegression Pipeline, DecisionTree, RandomForest, XGBoost)
# TODO 5.2: Set up StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# TODO 5.3: Run cross_validate across all models with return_train_score=True
# TODO 5.4: Build Leaderboard calculating Overfit Gap Δ = Train F1 - Val F1
# TODO 5.5: Flag any model with Δ > 0.08 as OVERFITTING

### Section 6: In-Depth Diagnostic Analysis (`src/evaluate.py`)
> **SRP**: Analyze error distributions (Type I vs Type II) via out-of-fold `cross_val_predict` and extract Gini feature importances.

In [ ]:
# TODO 6.1: Compute out-of-fold predictions using cross_val_predict on champion model
# TODO 6.2: Compute and visualize Confusion Matrix (TN, FP, FN, TP)
# TODO 6.3: Calculate Type I Error (False Positive) vs Type II Error (False Negative) counts
# TODO 6.4: Fit champion model on full X_train and plot top Gini feature importances

### Section 7: Hyperparameter Tuning & Model Comparison (`src/train.py` & `src/evaluate.py`)
> **OCP**: Systematically tune candidate models with parallel `GridSearchCV` (`n_jobs=-1`) and compare optimal parameters against each other.

In [ ]:
# TODO 7.1: Define parameter search grids for Logistic Regression, Decision Tree, Random Forest, XGBoost
# TODO 7.2: Run parallelized GridSearchCV (n_jobs=-1) across models
# TODO 7.3: Build comparative parameters table:
#          [Model, Best Parameters, Tuned CV F1, Baseline CV F1, Tuning Delta Δ, Rank]
# TODO 7.4: Identify the overall winning champion estimator

### Section 8: Strict Held-Out Evaluation & Artifact Persistence (`src/evaluate.py` & `src/train.py`)
> **Final Verification**: Evaluate the tuned champion on pristine unseen $X_{val}$ (179 rows) and serialize model weights with `joblib.dump`.

In [ ]:
# TODO 8.1: Predict on held-out validation set (y_val_pred and y_val_proba)
# TODO 8.2: Compute final held-out metrics (Accuracy, F1-Score, ROC-AUC)
# TODO 8.3: Print sklearn classification_report(y_val, y_val_pred)
# TODO 8.4: Serialize champion model artifact using joblib.dump(best_estimator, 'models/champion_rf.joblib')